# 6. Deployment
We rebuild the chosen homework solution on all eligible labelled purchases
and create the Kaggle CSV. This is a reproducible batch prediction process;
no web application is required.

The outer holdout was used for assessment, not a new model search. Final
fitting retains the model settings and feature-selection method chosen
inside development.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from preparation import fill_data, scale_data, encode_data

## Load the clean data
Clean training data include the former development, gap and holdout periods.
The 10 exact duplicates and 55 undated incomplete rows remain excluded.
Test keeps all purchase IDs in their original order.

In [2]:
train = pd.read_csv("../data/clean/train.csv", parse_dates=["purchase_date"])
test = pd.read_csv("../data/clean/test.csv", parse_dates=["purchase_date"])
target = "repeat_purchase_90d"
test_ids = test["ID"].copy()

train = train.sort_values(["purchase_date", "ID"])
print("Eligible training purchases:", len(train))
print("Test purchases:", len(test))
print("Days between last training and first test purchase:",
      (test["purchase_date"].min() - train["purchase_date"].max()).days)

Eligible training purchases: 6469
Test purchases: 986
Days between last training and first test purchase: 98


There are 6,469 eligible training purchases. The first test date is 98 days
after the last training purchase, exceeding the 90-day target window. All
retained training labels would therefore be known before test predictions.
The old 147-row gap can now contribute to final fitting.

## Available calendar inputs and target
We use the same feature definitions as in the earlier notebooks.

In [3]:
for df in [train, test]:
    df["purchase_year"] = df["purchase_date"].dt.year
    df["purchase_month"] = df["purchase_date"].dt.month.astype(str)
    df["purchase_weekday"] = df["purchase_date"].dt.dayofweek.astype(str)

train = train.set_index("ID")
test = test.set_index("ID")
y_train = train[target]
X_train = train.drop(columns=["customer_id", "purchase_date", target])
X_test = test.drop(columns=["customer_id", "purchase_date"])

## Refit learned preparation on all eligible training purchases
Medians, scaling parameters and category frequencies are learned again from
these training rows. Test only receives the transformations. The number of
one-hot columns can change because more training categories meet the threshold.

In [4]:
X_train, X_test, num, cat = fill_data(X_train, X_test)
X_train, X_test = scale_data(X_train, X_test, num)
X_train, X_test = encode_data(X_train, X_test, cat)
print("Full training inputs:", X_train.shape)
print("Test inputs:", X_test.shape)

Full training inputs: (6469, 302)
Test inputs: (986, 302)


## Refit L1 selection
Forest + L1 won the development comparison. We refit the same selector
(`C=0.1`) on all eligible training purchases. The encoded columns can differ
from development, so the selected list is learned again.

In [5]:
select = LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                             max_iter=2000, random_state=42)
select.fit(X_train, y_train)
keep = np.abs(select.coef_[0]) > 0.000001
cols = X_train.columns[keep].tolist()
print("Inputs used in final fitting:", len(cols))

Inputs used in final fitting: 58


In [6]:
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                random_state=42, n_jobs=2)
model.fit(X_train[cols], y_train)
pred = model.predict(X_test[cols])

## Build and check the submission
The CSV contains only the original purchase IDs and binary predictions.
Example labels in the Kaggle page and sample file are not ground truth.

In [7]:
sub = pd.DataFrame({"ID": test.index, target: pred.astype(int)})
print("Submission shape:", sub.shape)
print("Repeated IDs:", sub["ID"].duplicated().sum())
print("Predicted class counts:")
print(sub[target].value_counts().sort_index())
sub.head()

Submission shape: (986, 2)
Repeated IDs: 0
Predicted class counts:
repeat_purchase_90d
0    341
1    645
Name: count, dtype: int64


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1


In [8]:
sample = pd.read_csv("../data/raw/sample_submission.csv")
assert sub["ID"].tolist() == test_ids.tolist()
assert sub.columns.tolist() == sample.columns.tolist()
assert set(sub[target].unique()).issubset({0, 1})

## Save version 01
Re-running this notebook reproduces this version, using the fixed random seed.
Increment the version only when intentionally producing another solution.

In [9]:
os.makedirs("../submissions", exist_ok=True)
version = "01"
path = "../submissions/ML_Group_06_Version" + version + ".csv"
sub.to_csv(path, index=False)
pd.Series(cols, name="feature").to_csv("../results/final_features.csv", index=False)
print("Saved:", path)

Saved: ../submissions/ML_Group_06_Version01.csv


## Submission status
The CSV is generated locally. Uploading it to Kaggle and selecting a final
submission are separate actions and have not been performed here.

Use team `ML_Group_06`, keep the versioned filename and record the resulting
public F1 after submission. The public score uses 70% of test rows and may
differ from the chronological holdout; the remaining 30% form the private
leaderboard. The notebook and submitted CSV must describe the same solution.

The homework also requires a two-page PDF. The final project will extend
this work to at least five taught algorithms, optimisation and an additional
investigation.

The full refit produces 302 prepared inputs and selects 58. The CSV
contains 645 positive and 341 negative predictions. These counts describe
the predictions; they do not measure test performance.